# LaLonde Dataset: Direct Covariate Matching

This notebook demonstrates direct covariate matching (without propensity scores) on the LaLonde dataset to estimate the average treatment effect on the treated (ATT) of job training on earnings.

Unlike propensity score matching which reduces covariates to a single score, this approach matches treated and control units based on distance in the full covariate space.

In [1]:
import pandas as pd
import numpy as np
from causalml.match import NearestNeighborMatch
from dowhy import datasets
from sklearn.preprocessing import StandardScaler

In [2]:
# Load LaLonde dataset from DoWhy
data = datasets.lalonde_dataset()

print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nFirst few rows:")
data.head()

Dataset shape: (445, 12)

Columns: ['treat', 'age', 'educ', 'black', 'hisp', 'married', 'nodegr', 're74', 're75', 're78', 'u74', 'u75']

First few rows:


,treat,age,educ,black,hisp,married,nodegr,re74,re75,re78,u74,u75
0,False,23.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,0.00,1.0,1.0
1,False,26.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,12383.68,1.0,1.0
2,False,22.0,9.0,1.0,0.0,0.0,1.0,0.0,0.0,0.00,1.0,1.0
3,False,18.0,9.0,1.0,0.0,0.0,1.0,0.0,0.0,10740.08,1.0,1.0
4,False,45.0,11.0,1.0,0.0,0.0,1.0,0.0,0.0,11796.47,1.0,1.0


In [3]:
# Define treatment, outcome, and covariates
treatment_col = "treat"
outcome_col = "re78"
covariate_cols = ["age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75"]

# Standardize covariates for distance-based matching
# This ensures all variables are on the same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[covariate_cols])

# Add standardized covariates to dataframe
for i, col in enumerate(covariate_cols):
    data[f"{col}_scaled"] = X_scaled[:, i]

scaled_cols = [f"{col}_scaled" for col in covariate_cols]

In [5]:
# Perform nearest neighbor matching directly on scaled covariates
# This uses Euclidean distance in the covariate space
matcher = NearestNeighborMatch(caliper=3.0, replace=True, random_state=42)
matched_data = matcher.match(
    data=data, 
    treatment_col=treatment_col, 
    score_cols=scaled_cols
)

print(f"Original data: {len(data)} rows")
print(f"Matched data: {len(matched_data)} rows")
print(f"Treated units: {matched_data[treatment_col].sum()}")
print(f"Control units: {(~matched_data[treatment_col]).sum()}")

Original data: 445 rows
Matched data: 370 rows
Treated units: 185
Control units: 185


In [6]:
# Compute Average Treatment Effect on the Treated (ATT)
treated_outcome = matched_data.loc[matched_data[treatment_col] == 1, outcome_col].mean()
control_outcome = matched_data.loc[matched_data[treatment_col] == 0, outcome_col].mean()
att = treated_outcome - control_outcome

print(f"\nAverage outcome for treated: ${treated_outcome:,.2f}")
print(f"Average outcome for control: ${control_outcome:,.2f}")
print(f"\nEstimated ATT: ${att:,.2f}")


Average outcome for treated: $6,349.14
Average outcome for control: $4,533.48

Estimated ATT: $1,815.66


In [ ]:
# Compare with true experimental effect
# The LaLonde dataset comes from a randomized experiment (NSW demonstration)
# so we can compute the "true" treatment effect from the experimental data
true_treated_mean = data.loc[data[treatment_col] == 1, outcome_col].mean()
true_control_mean = data.loc[data[treatment_col] == 0, outcome_col].mean()
true_ate = true_treated_mean - true_control_mean

print(f"\n{'='*60}")
print(f"Comparison with Experimental Benchmark")
print(f"{'='*60}")
print(f"True experimental ATE:      ${true_ate:,.2f}")
print(f"Covariate matching ATT:     ${att:,.2f}")
print(f"Difference:                 ${att - true_ate:,.2f} ({(att - true_ate) / true_ate * 100:.1f}%)")

In [7]:
# Check covariate balance after matching
print("\nCovariate Balance (Standardized Mean Differences):")
print("=" * 60)

for col in covariate_cols:
    treated_mean = matched_data.loc[matched_data[treatment_col] == 1, col].mean()
    control_mean = matched_data.loc[matched_data[treatment_col] == 0, col].mean()
    pooled_sd = np.sqrt(
        (matched_data.loc[matched_data[treatment_col] == 1, col].var() + 
         matched_data.loc[matched_data[treatment_col] == 0, col].var()) / 2
    )
    smd = (treated_mean - control_mean) / pooled_sd if pooled_sd > 0 else 0
    print(f"{col:12s}: {smd:7.4f}")


Covariate Balance (Standardized Mean Differences):
age         :  0.1176
educ        : -0.0114
black       :  0.0000
hisp        :  0.0000
married     :  0.0420
nodegr      :  0.0000
re74        :  0.0518
re75        :  0.1259
